Implementing the fixed-feature extractor model

In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow import keras
import math
import numpy as np
from tensorflow.keras import optimizers
from tensorflow.keras.optimizers import Adam

import wandb
from wandb.integration.keras import WandbMetricsLogger

Img_channels = 3  # R, G, B channels

Img_class = ['daisy', 'dandelion', 'roses', 'sunflowers', 'tulips']

# Start a run, tracking hyperparameters

sweep_config = {'method': 'grid',
                'metric': {'name': 'val_accuracy', 'goal': 'maximize'},
                'parameters':
                    {'Epochs': {'values': [10]},
                     'batch_size': {'values': [8]},
                     'img_size': {'values': [128]},
                     'learning_rate': {'values': [0.0001]},
                     'hidden_units': {'values': [128]}}}

sweep_id = wandb.sweep(sweep_config, project="Flower_classification_TL")

def train():
    with wandb.init() as run:
      config = wandb.config
      Img_size = config.img_size
      Batch_size = config.batch_size
      Epochs = config.Epochs
      Learning_rate = config.learning_rate
      Hidden_units = config.hidden_units

      def read_and_decode(filename, resize_dims):
          # Read the raw file
          img_df = tf.io.read_file(filename)

          # Decode image
          img = tf.image.decode_jpeg(img_df, channels=Img_channels)

          # Convert pixels to floats
          img = tf.cast(img, tf.float32)
          #img = tf.image.convert_image_dtype(img, tf.float32)

          # Resize image
          img = tf.image.resize(img, resize_dims)

          # Scale to [-1,1] as it is required by MobileNet V2's imagenet weights
          img = tf.keras.applications.mobilenet_v2.preprocess_input(img)

          return img

      def parse_csv_line(csv_line):
          record_default = ["", ""]  # Specify the data type for each column
          filename, label_string = tf.io.decode_csv(csv_line, record_default)

          # Load the image
          img = read_and_decode(filename, [Img_size, Img_size])

          # Convert label string to integer based on the Img_class index (0-4)
          label = tf.argmax(tf.math.equal(label_string, Img_class))

          return img, label

      # Training and validation datasets
      train_dataset = (tf.data.TextLineDataset("gs://practical-ml-vision-book-data/flowers_5_jpeg/flower_photos/train_set.csv")
                      .map(parse_csv_line, num_parallel_calls=tf.data.AUTOTUNE)
                      .batch(Batch_size)
                      .prefetch(tf.data.AUTOTUNE))

      eval_dataset = (tf.data.TextLineDataset("gs://practical-ml-vision-book-data/flowers_5_jpeg/flower_photos/eval_set.csv")
                      .map(parse_csv_line, num_parallel_calls=tf.data.AUTOTUNE)
                      .batch(Batch_size)
                      .prefetch(tf.data.AUTOTUNE))

      #Load pre-trained model
      base_model = tf.keras.applications.MobileNetV2(input_shape=(Img_size, Img_size, Img_channels), include_top=False, weights='imagenet') #include_top = False removes the classification head, load weights trained on imagenet

      #Freeze the base model
      base_model.trainable = False #ImageNet weights will not get modified during training

      #Add the new classification head
      model = tf.keras.Sequential([
          base_model,
          tf.keras.layers.GlobalAveragePooling2D(),
          keras.layers.BatchNormalization(),
          tf.keras.layers.Dense(Hidden_units, activation = 'relu'),
          tf.keras.layers.Dense(len(Img_class), activation='softmax') #5-layers since we are using the 5-layer dataset
      ])

      # Compile model
      model.compile(
            optimizer=keras.optimizers.Adam(learning_rate=config.learning_rate),
            loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
            metrics=["accuracy"]
        )
      # Visualize model
      model.summary()

      keras.utils.plot_model(model, show_shapes=True, show_layer_names=True, to_file="model_visualization.png")

        # Train
      callbacks = [WandbMetricsLogger(log_freq=5)]

      model.fit(
            train_dataset,
            validation_data=eval_dataset,
            epochs=config.Epochs,
            callbacks=callbacks
        )


In [ ]:
# Launch W&B agent
wandb.agent(sweep_id, function=train)